# Lesson 16: SmolVLA execution and inputs

This lesson uses small code cells to show the real SmolVLA path. Run cells in order. This notebook contains complete exercise answers.

**Result:** a camera transform and tokenized prompt that match the published SmolVLA input contract.

**Notebook pair:** [Learner notebook](16-smolvla-execution.ipynb).

**Evidence boundary:** CPU checks show tensor mechanics. They do not show a useful robot policy. Cells with an experiment tag load local model or LIBERO assets and can take much longer.

In [ ]:
import os
import sys
from pathlib import Path


def find_workspace() -> Path:
    for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
        if (path / "manifests" / "act_references.yaml").is_file():
            return path
    raise FileNotFoundError("Start Jupyter inside the tutorial workspace.")


WORKSPACE = find_workspace()
os.environ["HF_HUB_CACHE"] = str(WORKSPACE / ".local-data/hf/hub")
os.environ["HF_DATASETS_CACHE"] = str(WORKSPACE / ".local-data/hf/datasets")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"
os.environ.setdefault("TORCH_HOME", str(WORKSPACE / ".local-data/torch"))
if str(WORKSPACE / "src") not in sys.path:
    sys.path.insert(0, str(WORKSPACE / "src"))
CONFIGS = WORKSPACE / "configs"
DATA = WORKSPACE / ".local-data"

In [ ]:
from collections.abc import Sequence

import torch
import torch.nn.functional as F
import yaml
from torch.utils.data import default_collate
from transformers import AutoTokenizer

from vla_lab.sim.episode import open_libero_task, run_policy_episode
from vla_lab.teaching.act_model import frame_offset_seconds, open_libero_dataset
from vla_lab.teaching.interfaces import ProcessorObservation, PromptTokenizer, VLAInputs
from vla_lab.teaching.smolvla import CallbackSmolVLAPolicy, SmolVLAAdapter, load_prepared_smolvla

## Model map: observations become a cached prefix

SmolVLA receives camera images, robot state, and an instruction. It turns image patches and text tokens into a **prefix**. The VLM builds key and value tensors for that prefix. The action expert then reads this context and predicts a chunk of robot actions.

```text
front image (B,3,H,W) ── vision encoder ─┐
wrist image (B,3,H,W) ── vision encoder ─┼─ VLM prefix (B,Lp,C) ── key/value cache
instruction ── tokenizer ────────────────┤                          │
robot state (B,8) ────────────────────────┘                          ▼
noisy action chunk (B,T,32) + time (B,) ─────────────── action expert ── velocity
```

`B` is batch size (number of examples), `H,W` are image pixels, `Lp` is prefix-token count, `C` is hidden width, and `T` is action steps. The action feature has 7 values per step: 3 normalized relative position commands, 3 normalized relative rotation commands, and 1 normalized gripper command. The simulator maps these bounded values to controller movement. State has 8 standardized values, so its numeric units are training-set standard deviations. The action feature has 7 normalized relative controller commands, bounded by the processor; they are not metres or radians. Time is a unitless fraction from 0 to 1. SmolVLA pads each row to 32 values for its expert. The attention mask has `True` for a real token and `False` for padding.

A model does not understand pixels as raw arrays. The vision encoder expects a fixed square image in `[-1,1]`. Fit the image without changing its shape: scale it to fit, put zero-valued image padding at the top or left, then map `[0,1]` to `[-1,1]`. Thus padding becomes `-1`. A text tokenizer maps a prompt to integer token IDs; a mask says which IDs are real. Newline is part of the trained prompt format.

The published checkpoint is a 450M-parameter SmolVLA. For the asset experiment, this notebook uses the published `smolvla-libero` weights and the local SmolVLM2 tokenizer. It does not download files. A missing local file is a setup error, not a reason to use random weights.

## Exercise 16.1 — Build model-ready inputs

**Purpose.** Implement the camera transform used by the vision encoder. The result must preserve aspect ratio and produce values in the encoder's expected range.

**Inputs and output.** `image` has shape `(B,3,h,w)` and `float32` values in `[0,1]`. `height,width` are output pixels. Return `(B,3,height,width)` in `[-1,1]`. Resize with bilinear interpolation. Do not stretch the image. Put padding at the top or left. Padding starts at `0` before normalization, so it becomes `-1`.

**Example.** For a white image of shape `(1,3,32,64)` and output `(64,64)`, the resized image is `32×64`. It fills the lower half; the upper half is padding. The output contains only `-1` and `1`.

Also implement `encode_instruction`. The tokenizer receives a list of strings. Add a final newline to each prompt when needed. Use right padding and truncation at `max_length`. Return token IDs as `torch.long` and the attention mask as `torch.bool`; `True` means the token is real. For prompts of token lengths 3 and 1, the padded mask is `[True, True, True]` and `[True, False, False]`.

In [ ]:
def fit_image(image: torch.Tensor, height: int, width: int) -> torch.Tensor:
    current_height, current_width = image.shape[-2:]
    ratio = max(current_width / width, current_height / height)
    resized_height = int(current_height / ratio)
    resized_width = int(current_width / ratio)
    if (resized_height, resized_width) != (current_height, current_width):
        image = F.interpolate(
            image, size=(resized_height, resized_width), mode="bilinear", align_corners=False
        )
    image = F.pad(image, (width - resized_width, 0, height - resized_height, 0), value=0.0)
    return image * 2.0 - 1.0


def encode_instruction(
    tokenizer: PromptTokenizer, prompts: str | Sequence[str], max_length: int
) -> tuple[torch.Tensor, torch.Tensor]:
    texts = [prompts] if isinstance(prompts, str) else list(prompts)
    texts = [prompt if prompt.endswith("\n") else prompt + "\n" for prompt in texts]
    encoded = tokenizer(
        texts,
        max_length=max_length,
        truncation=True,
        padding="longest",
        padding_side="right",
        return_tensors="pt",
    )
    return encoded["input_ids"].to(torch.long), encoded["attention_mask"].to(torch.bool)


class ToyTokenizer:
    """Small deterministic tokenizer for CPU checks; the experiment loads the real tokenizer."""

    def __call__(
        self: "ToyTokenizer",
        texts: Sequence[str],
        max_length: int,
        truncation: bool,
        padding: str,
        padding_side: str,
        return_tensors: str,
    ) -> dict[str, torch.Tensor]:
        rows = [[len(word) for word in text.split()][:max_length] for text in texts]
        width = max(len(row) for row in rows)
        ids = [row + [0] * (width - len(row)) for row in rows]
        masks = [[True] * len(row) + [False] * (width - len(row)) for row in rows]
        return {"input_ids": torch.tensor(ids), "attention_mask": torch.tensor(masks)}


wide = torch.ones(1, 3, 32, 64)
result = fit_image(wide, 64, 64)
assert result.shape == (1, 3, 64, 64)
assert result[:, :, :32].eq(-1).all() and result[:, :, 32:].eq(1).all()
tokens, mask = encode_instruction(ToyTokenizer(), ["move the cup", "cup"], 8)
assert tokens.dtype == torch.long and mask.dtype == torch.bool
assert mask.tolist() == [[True, True, True], [True, False, False]]

The next untagged cells check your math with CPU tensors. The full experiment then compares the learner image transform with the published policy preprocessor on real local LIBERO images and runs one published SmolVLA action prediction.

In [ ]:
wide = torch.ones(2, 3, 32, 64)
small = fit_image(wide, 64, 64)
assert small.shape == (2, 3, 64, 64)
assert small[:, :, :32].eq(-1).all() and small[:, :, 32:].eq(1).all()
tall = fit_image(torch.ones(1, 3, 64, 32), 64, 64)
assert tall[:, :, :, :32].eq(-1).all() and tall[:, :, :, 32:].eq(1).all()
tokens, mask = encode_instruction(ToyTokenizer(), ["move the cup", "cup"], 8)
print("CPU image and token checks passed; IDs:", tokens.tolist(), "mask:", mask.tolist())

**Explain (2–3 sentences).** Why does padding become `-1` after normalization? What can a one-step output check prove, and what would require a LIBERO rollout?

In [ ]:
# Real published SmolVLA execution: local checkpoint and local LIBERO frames.
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
POLICY, PREPROCESSOR, POSTPROCESSOR = load_prepared_smolvla("published", DATA, device=DEVICE)
PROMPT = "put the bowl on the plate"
ACTION_CHUNK_SIZE = POLICY.config.chunk_size
REAL_DATASET = open_libero_dataset(
    CONFIGS,
    DATA,
    "bowl_plate",
    "validation",
    {"action": frame_offset_seconds(list(range(ACTION_CHUNK_SIZE)))},
)
REAL = default_collate([REAL_DATASET[0]])
REAL["task"] = [PROMPT]
print("Loaded published SmolVLA on", DEVICE, "and one raw local LIBERO window")

In [ ]:
# The policy input pipeline turns this observation and instruction into model tensors.
OBSERVATION = {
    "observation.images.image": REAL["observation.images.image"],
    "observation.images.image2": REAL["observation.images.image2"],
    "observation.state": REAL["observation.state"],
    "task": [PROMPT],
}
# The preprocessor adds normalized state, token IDs, and masks expected by the checkpoint.
MODEL_BATCH = PREPROCESSOR(OBSERVATION)
TOKENIZER = AutoTokenizer.from_pretrained(str(DATA / "models/smolvlm2-500m-video-instruct"))


def learner_input_fn(adapter: SmolVLAAdapter, batch: ProcessorObservation) -> VLAInputs:
    inputs = adapter.pinned_inputs(batch)
    width, height = adapter.config.resize_imgs_with_padding
    inputs["images"] = [
        fit_image(batch[key], height, width) for key in adapter.config.image_features
    ]
    inputs["img_masks"] = [
        torch.ones(inputs["state"].shape[0], dtype=torch.bool, device=inputs["state"].device)
        for _ in inputs["images"]
    ]
    tokens, mask = encode_instruction(TOKENIZER, batch["task"], adapter.config.tokenizer_max_length)
    inputs["lang_tokens"] = tokens.to(inputs["state"].device)
    inputs["lang_masks"] = mask.to(inputs["state"].device)
    return inputs


CALLBACK = CallbackSmolVLAPolicy(POLICY, input_fn=learner_input_fn, num_steps=4)
with torch.no_grad():
    OUTPUT = CALLBACK.select_action(MODEL_BATCH)
print("action shape:", tuple(OUTPUT.shape), "| finite:", bool(torch.isfinite(OUTPUT).all()))
SPLITS = yaml.safe_load((CONFIGS / "eval" / "splits.yaml").read_text())
STATE_ID = SPLITS["tasks"]["bowl_plate"]["development_state_ids"][0]
ENVIRONMENT = open_libero_task("bowl_plate", STATE_ID, CONFIGS, DATA, control_hz=20)
try:
    EPISODE = run_policy_episode(
        ENVIRONMENT,
        CALLBACK,
        PREPROCESSOR,
        POSTPROCESSOR,
        "bowl_plate",
        STATE_ID,
        CONFIGS,
        task_prompt=PROMPT,
        reset_seed=20261006 + STATE_ID,
        control_hz=20,
    )
finally:
    ENVIRONMENT.close()
print({"success": EPISODE.success, "steps": EPISODE.steps, "state_id": STATE_ID, "control_hz": 20})

The dataset window selects 50 adjacent action rows by frame offset. The declared 10 FPS storage label supplies index timestamps; it does not define action duration. LIBERO sends one relative controller action per 20 Hz step. The runner does not repeat or resample dataset labels.